# Multiclass Classification with a Feed-Forward Network

In the previous notebook the network predicted one number. Here it chooses one of seven classes. The data is the [Forest Covertype](https://archive.ics.uci.edu/dataset/31/covertype) dataset from the US Forest Service: each row is a 30 x 30 metre patch of forest in Colorado, described by its terrain, and the target is the tree species that covers it. The data is complete and already numeric, so all of your attention can go to the network itself.

## Learning Objectives

By the end of this notebook, you should be able to:

- Split a tabular dataset into training, validation and test sets and compare against a majority-class baseline.
- Build a Keras Sequential classifier with a softmax output layer.
- Train the classifier with sparse categorical cross-entropy and plot its learning curves.
- Evaluate the test accuracy and identify the most confused classes in a confusion matrix.

## Setup

In [ ]:
import keras
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import fetch_covtype
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

keras.utils.set_random_seed(42)

## 1. Load and Explore the Data

[`fetch_covtype`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_covtype.html) downloads the data on the first run, which takes about a minute, and caches it afterwards. The full dataset has 581,012 rows. A random sample of 60,000 rows is enough to train a good model and keeps every training run under a minute.

In [ ]:
covtype = fetch_covtype(as_frame=True)
forest = covtype.frame.sample(60_000, random_state=42)

print("Rows and columns:", forest.shape)
forest.iloc[:3, :12]

There are 54 features:

- **10 numeric terrain measurements**, such as `Elevation` (metres), `Slope` (degrees) and the distances to the nearest water, road and wildfire ignition point.
- **4 `Wilderness_Area` columns** and **40 `Soil_Type` columns**. These are categorical variables that are already one-hot encoded: in each group exactly one column is 1.

The target `Cover_Type` holds the codes 1 to 7. Keras expects class labels that start at 0, so we subtract 1 and keep a list of the species names in the same order.

In [ ]:
class_names = [
    "Spruce/Fir", "Lodgepole Pine", "Ponderosa Pine", "Cottonwood/Willow",
    "Aspen", "Douglas-fir", "Krummholz",
]

X = forest.drop(columns="Cover_Type")
y = forest["Cover_Type"] - 1  # codes 1-7 become labels 0-6

class_shares = y.value_counts(normalize=True).sort_index()
class_shares.index = class_names
class_shares.round(3).to_frame("Share of rows")

The classes are strongly imbalanced. Lodgepole Pine (0.486) and Spruce/Fir (0.367) together cover 85 % of the rows, while Cottonwood/Willow covers only 0.4 %. A model can reach a high overall accuracy by getting the two large classes right and neglecting the rest, so later in the notebook we also look at the accuracy of each class.

## 2. Split the Data and Set Baselines

As in the previous notebook, split the data with [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) into a training set, a validation set to watch training and a test set for the final comparison. `stratify=y` keeps the class shares the same in every split, which matters when some classes are rare.

Two baselines show the scores the network has to exceed. The **majority-class baseline** always predicts the most common species. **Logistic regression**, which you met in week 2, is a linear model: it is exactly a single layer of neurons with a softmax output and no hidden layers.

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
)
print("Train:", X_train.shape, " Validation:", X_valid.shape, " Test:", X_test.shape)

In [ ]:
test_accuracy = {"Majority class": (y_test == y_train.mode()[0]).mean()}

log_reg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
log_reg.fit(X_train, y_train)
test_accuracy["Logistic regression"] = log_reg.score(X_test, y_test)

pd.Series(test_accuracy).round(3).to_frame("Test accuracy")

Always predicting Lodgepole Pine is right on 48.6 % of the test rows, so any useful model has to exceed that. Logistic regression reaches 0.724. A linear model can only draw straight boundaries between the classes; the hidden layers of a neural network can bend them.

## 3. Build the Classifier

The network takes the 54 features, standardises them with a [`Normalization`](https://keras.io/api/layers/preprocessing_layers/numerical/normalization/) layer as in notebook 03, passes them through two hidden layers, and ends with one neuron per class.

<img src="../assets/forest_cover_classifier.png" width="900">

The output layer uses the **softmax** activation. It turns the seven raw outputs into seven probabilities that are all positive and sum to 1, so the network's answer is the class with the highest probability.

In [ ]:
norm_layer = keras.layers.Normalization()
norm_layer.adapt(X_train.to_numpy())  # learn the mean and variance of each feature

model = keras.Sequential(
    [
        keras.Input(shape=(X_train.shape[1],)),
        norm_layer,
        keras.layers.Dense(units=128, activation="relu"),
        keras.layers.Dense(units=64, activation="relu"),
        keras.layers.Dense(units=7, activation="softmax"),  # one probability per species
    ]
)
model.summary()

The first hidden layer connects each of the 54 features to each of its 128 neurons: $54 \times 128 + 128 = 7{,}040$ parameters. The second has $128 \times 64 + 64 = 8{,}256$ and the output layer $64 \times 7 + 7 = 455$.

## 4. Compile and Train

Compared with regression, three settings change:

| | Regression (notebook 03) | Classification (this notebook) |
|---|---|---|
| Output layer | 1 neuron, no activation | 7 neurons, softmax |
| Loss | mean squared error | sparse categorical cross-entropy |
| Metric | RMSE | accuracy |

**Cross-entropy** is small when the model gives a high probability to the correct class and grows quickly when it is confidently wrong. The *sparse* version accepts the labels as integers (0 to 6). If the labels were one-hot encoded vectors you would use `categorical_crossentropy` instead.

`batch_size=128` sets how many rows the model sees before each weight update. Larger batches mean fewer, smoother updates per epoch and a faster epoch.

In [ ]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"],
)
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=128,
    validation_data=(X_valid, y_valid),
    verbose=0,
)

In [ ]:
history_df = pd.DataFrame(history.history)
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
history_df[["loss", "val_loss"]].plot(ax=ax_loss, xlabel="Epoch", title="Loss", grid=True)
history_df[["accuracy", "val_accuracy"]].plot(
    ax=ax_acc, xlabel="Epoch", title="Accuracy", grid=True
)
plt.show()

history_df.tail(3).round(3)

After 30 epochs the training accuracy is 0.847 and the validation accuracy is 0.819, and both are still improving slowly. The validation loss (0.434) is above the training loss (0.371): the model is a little better on rows it has seen than on new ones. That gap is the first sign of overfitting, which W5D2 covers.

## 5. Evaluate and Predict

In [ ]:
test_loss, test_accuracy["Feed-forward network"] = model.evaluate(X_test, y_test, verbose=0)
pd.Series(test_accuracy).round(3).to_frame("Test accuracy")

The network reaches a test accuracy of 0.819, compared with 0.724 for logistic regression and 0.486 for the majority-class baseline. Logistic regression misclassifies 27.6 % of the test rows and the network 18.1 %, so the hidden layers remove about a third of the linear model's errors.

`predict()` returns seven probabilities per row. `argmax` picks the class with the highest one.

In [ ]:
probabilities = model.predict(X_test, verbose=0)
y_pred = probabilities.argmax(axis=1)

first_rows = pd.DataFrame(probabilities[:3].round(2), columns=class_names)
first_rows["Actual"] = [class_names[label] for label in y_test.iloc[:3]]
first_rows

All three rows are Lodgepole Pine, and the model predicts that class each time with a probability between 0.81 and 0.89. The other probabilities show which alternatives the model considered: 0.17 for Spruce/Fir in the third row, and 0.10 for Aspen in the second.

### Which Classes Get Confused?

Accuracy is a single number, and with imbalanced classes it is dominated by the two large ones. A confusion matrix, drawn here with [`ConfusionMatrixDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html), shows which classes the model mixes up. Each row is the true class, and the values are normalised so that each row sums to 1.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=class_names,
    normalize="true",
    values_format=".2f",
    xticks_rotation=45,
    colorbar=False,
    ax=ax,
)
plt.show()

The matrix is easier to read alongside two summaries: the accuracy for each class, and the most frequent mistakes.

In [ ]:
results = pd.DataFrame(
    {
        "Actual": [class_names[i] for i in y_test],
        "Predicted": [class_names[i] for i in y_pred],
    }
)
results["Correct"] = results["Actual"] == results["Predicted"]

per_class = results.groupby("Actual")["Correct"].agg(["mean", "size"])
per_class.columns = ["Accuracy", "Test rows"]
per_class.sort_values("Accuracy").round(2)

In [ ]:
mistakes = results[~results["Correct"]]
mistakes[["Actual", "Predicted"]].value_counts().head(5).to_frame("Count")

The overall accuracy of 0.819 hides large differences between the classes. The two common species reach 0.82 (Spruce/Fir) and 0.86 (Lodgepole Pine), but the rare ones are much weaker: Cottonwood/Willow 0.38 (only 53 test rows), Aspen 0.47 and Douglas-fir 0.51. With few training examples the model learns little about these classes, and predicting a common class lowers the loss more often than not.

The most frequent mistakes are between the two largest classes, in both directions: 733 Spruce/Fir rows predicted as Lodgepole Pine and 714 the other way round. Among the smaller classes, Douglas-fir is most often mistaken for Ponderosa Pine (125 rows). Class weights are one way to give rare classes more influence on the loss.

## Try It: Change the Network

Build a new classifier with a different architecture or training setting, for example fewer or more neurons, a third hidden layer or more epochs. Compare its **validation** accuracy with the main model's. The test set is kept for the final comparison, so the Try It does not use it. The normalisation layer is already fitted, so you can reuse it.

In [ ]:
keras.utils.set_random_seed(42)
valid_accuracy = model.evaluate(X_valid, y_valid, verbose=0)[1]  # the main model on the validation set

# A wider and deeper network trained for longer
hidden_layers = [
    keras.layers.Dense(units=256, activation="relu"),
    keras.layers.Dense(units=128, activation="relu"),
    keras.layers.Dense(units=64, activation="relu"),
]
n_epochs = 40

model_try = keras.Sequential(
    [
        keras.Input(shape=(X_train.shape[1],)),
        norm_layer,
        *hidden_layers,
        keras.layers.Dense(units=7, activation="softmax"),
    ]
)
model_try.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"],
)
history_try = model_try.fit(
    X_train, y_train,
    epochs=n_epochs,
    batch_size=128,
    validation_data=(X_valid, y_valid),
    verbose=0,
)

In [ ]:
valid_accuracy_try = model_try.evaluate(X_valid, y_valid, verbose=0)[1]
print(f"Validation accuracy: {valid_accuracy_try:.3f} (main model: {valid_accuracy:.3f})")

## Summary

- A classifier ends in one neuron per class with a softmax activation, which outputs a probability for every class.
- Sparse categorical cross-entropy is the loss for integer class labels, and accuracy is the usual metric.
- Baselines put the accuracy in context: a majority-class guess and a linear model show how much the hidden layers add.
- With imbalanced classes, per-class accuracy and the confusion matrix show failures that the overall accuracy hides.
- The gap between training and validation curves is the first sign of overfitting, which W5D2 covers.

## References

- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd edition, chapter 10.
- Chollet, F. and Watson, M. (2025). *Deep Learning with Python*, 3rd edition, chapter 4: Classification and regression. [Free to read online](https://deeplearningwithpython.io/).
- Blackard, J. (1998). [Covertype dataset](https://archive.ics.uci.edu/dataset/31/covertype). UCI Machine Learning Repository.
- Keras guide: [The Sequential model](https://keras.io/guides/sequential_model/).